# AAI614: Data Science & its Applications

*Notebook 3.1: Practice with Data Collections*

<a href="https://colab.research.google.com/github/harmanani/AAI614/blob/main/Week%203/Notebook3.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Source: Scraping with Python http://shop.oreilly.com/product/0636920034391.do

In [49]:
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

# Add a User-Agent header to mimic a browser request
req = Request('http://en.wikipedia.org/wiki/Kevin_Bacon', headers={'User-Agent': 'Mozilla/5.0'})
html = urlopen(req)
bs = BeautifulSoup(html, 'html.parser')
for link in bs.find_all('a'):
    if 'href' in link.attrs:
        print(link.attrs['href'])

#bodyContent
/wiki/Main_Page
/wiki/Wikipedia:Contents
/wiki/Portal:Current_events
/wiki/Special:Random
/wiki/Wikipedia:About
//en.wikipedia.org/wiki/Wikipedia:Contact_us
/wiki/Help:Contents
/wiki/Help:Introduction
/wiki/Wikipedia:Community_portal
/wiki/Special:RecentChanges
/wiki/Wikipedia:File_upload_wizard
/wiki/Special:SpecialPages
/wiki/Main_Page
/wiki/Special:Search
https://donate.wikimedia.org/?wmf_source=donate&wmf_medium=sidebar&wmf_campaign=en.wikipedia.org&uselang=en
/w/index.php?title=Special:CreateAccount&returnto=Kevin+Bacon
/w/index.php?title=Special:UserLogin&returnto=Kevin+Bacon
https://donate.wikimedia.org/?wmf_source=donate&wmf_medium=sidebar&wmf_campaign=en.wikipedia.org&uselang=en
/w/index.php?title=Special:CreateAccount&returnto=Kevin+Bacon
/w/index.php?title=Special:UserLogin&returnto=Kevin+Bacon
#
#Early_life_and_education
#Acting_career
#Early_work
#1980s
#1990s
#2000s
#2010s
#Other_ventures
#Six_Degrees_of_Kevin_Bacon
#Personal_life
#Accolades
#Awards_and_nomin

## Retrieving Articles Only

In [51]:
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import re

# Add a User-Agent header to mimic a browser request
req = Request('http://en.wikipedia.org/wiki/Kevin_Bacon', headers={'User-Agent': 'Mozilla/5.0'})
html = urlopen(req)
bs = BeautifulSoup(html, 'html.parser')
for link in bs.find('div', {'id':'bodyContent'}).find_all(
    'a', href=re.compile('^(/wiki/)((?!:).)*$')):
    print(link.attrs['href'])

In [54]:
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import re

# Add a User-Agent header to mimic a browser request
req = Request('http://en.wikipedia.org/wiki/Kevin_Bacon', headers={'User-Agent': 'Mozilla/5.0'})
html = urlopen(req)
bs = BeautifulSoup(html, 'html.parser')
for link in bs.find('div', {'id':'bodyContent'}).find_all(
    'a', href=re.compile('^(/wiki/)((?!:).)*$')):
    print(link.attrs['href'])

## Random Walk

In [57]:
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import datetime
import random
import re

random.seed(datetime.datetime.now().strftime('%s'))
def getLinks(articleUrl):
    # Add a User-Agent header to mimic a browser request
    req = Request(f'http://en.wikipedia.org{articleUrl}', headers={'User-Agent': 'Mozilla/5.0'})
    html = urlopen(req)
    bs = BeautifulSoup(html, 'html.parser')
    return bs.find('div', {'id':'bodyContent'}).find_all('a', href=re.compile('^(/wiki/)((?!:).)*$'))

links = getLinks('/wiki/Kevin_Bacon')
while len(links) > 0:
    newArticle = links[random.randint(0, len(links)-1)].attrs['href']
    print(newArticle)
    links = getLinks(newArticle)

## Recursively crawling an entire site

In [72]:
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import re

pages = set()
MAX_PAGES = 15

def getLinks(pageUrl):
    if len(pages) >= MAX_PAGES:
        return
    req = Request(f'http://en.wikipedia.org{pageUrl}', headers={'User-Agent': 'Mozilla/5.0'})
    html = urlopen(req)
    bs = BeautifulSoup(html, 'html.parser')
    for link in bs.find_all('a', href=re.compile('^(/wiki/)')):
        if len(pages) >= MAX_PAGES:
            return
        if 'href' in link.attrs:
            if link.attrs['href'] not in pages:
                #We have encountered a new page
                newPage = link.attrs['href']
                print(newPage)
                pages.add(newPage)
                getLinks(newPage)

getLinks('')
print(f"\nStopped after {len(pages)} pages.")

/wiki/Main_Page
/wiki/Wikipedia:Contents
/wiki/Portal:Current_events
/wiki/Special:Random
/wiki/Wikipedia:About
/wiki/Help:Contents
/wiki/Help:Introduction
/wiki/Wikipedia:Community_portal
/wiki/Special:RecentChanges
/wiki/Wikipedia:File_upload_wizard
/wiki/Special:SpecialPages
/wiki/Special:Search
/wiki/Help:Searching
/wiki/Help_talk:Searching
/wiki/Special:WhatLinksHere/Help_talk:Searching

Stopped after 15 pages.


## Collecting Data Across an Entire Site

In [63]:
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import re

pages = set()
def getLinks(pageUrl):
    # Add a User-Agent header to mimic a browser request
    req = Request(f'http://en.wikipedia.org{pageUrl}', headers={'User-Agent': 'Mozilla/5.0'})
    html = urlopen(req)
    bs = BeautifulSoup(html, 'html.parser')
    try:
        print(bs.h1.get_text())
        #mw-parser-output
        bodyContent = bs.find('div', {'id':'bodyContent'}).find_all('p')
        if len(bodyContent):
            print(bodyContent[0])
        print(bs.find(id='ca-edit').find('a').attrs['href'])
    except AttributeError:
        print('This page is missing something! Continuing.')

    for link in bs.find_all('a', href=re.compile('^(/wiki/)')):
        if 'href' in link.attrs:
            if link.attrs['href'] not in pages:
                #We have encountered a new page
                newPage = link.attrs['href']
                print('-'*20)
                print(newPage)
                pages.add(newPage)
                getLinks(newPage)
getLinks('/wiki/General-purpose_programming_language')

General-purpose programming language
<p id="mwAw">In <a class="mw-redirect" href="https://en.wikipedia.org/wiki/Computer_software" id="mwBA" rel="mw:WikiLink" title="Computer software">computer software</a>, a <b id="mwBQ">general-purpose programming language</b> (<b id="mwBg">GPL</b>) is a <a href="https://en.wikipedia.org/wiki/Programming_language" id="mwBw" rel="mw:WikiLink" title="Programming language">programming language</a> for building <a href="https://en.wikipedia.org/wiki/Software" id="mwCA" rel="mw:WikiLink" title="Software">software</a> in a wide variety of application <a href="https://en.wikipedia.org/wiki/Domain_(software_engineering)" id="mwCQ" rel="mw:WikiLink" title="Domain (software engineering)">domains</a>. Conversely, a <a href="https://en.wikipedia.org/wiki/Domain-specific_language" id="mwCg" rel="mw:WikiLink" title="Domain-specific language">domain-specific programming language</a> (DSL) is used within a specific area. For example, <a href="https://en.wikipedia.o

HTTPError: HTTP Error 404: Not Found

## Crawling across the Internet

In [70]:
from urllib.request import urlopen, Request
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import re
import datetime
import random
import time # Import the time module

#Retrieves a list of all Internal links found on a page
def getInternalLinks(bs, url):
    netloc = urlparse(url).netloc
    scheme = urlparse(url).scheme
    internalLinks = set()
    for link in bs.find_all('a'):
        if not link.attrs.get('href'):
            continue
        parsed = urlparse(link.attrs['href'])
        if parsed.netloc == '':
            internalLinks.add(f'{scheme}://{netloc}/{link.attrs["href"].strip("/")}')
        elif parsed.netloc == netloc:
            internalLinks.add(link.attrs['href'])
    return list(internalLinks)

#Retrieves a list of all external links found on a page
def getExternalLinks(bs, url):
    netloc = urlparse(url).netloc
    externalLinks = set()
    for link in bs.find_all('a'):
        if not link.attrs.get('href'):
            continue
        parsed = urlparse(link.attrs['href'])
        if parsed.netloc != '' and parsed.netloc != netloc:
            externalLinks.add(link.attrs['href'])
    return list(externalLinks)

def getRandomExternalLink(startingPage):
    # Add a User-Agent header to mimic a browser request
    req = Request(startingPage, headers={'User-Agent': 'Mozilla/5.0'})
    time.sleep(random.uniform(1, 3)) # Add a random delay between 1 and 3 seconds
    bs = BeautifulSoup(urlopen(req), 'html.parser')
    externalLinks = getExternalLinks(bs, startingPage)
    if not len(externalLinks):
        print('No external links, looking around the site for one')
        internalLinks = getInternalLinks(bs, startingPage)
        return getRandomExternalLink(random.choice(internalLinks))
    else:
        return random.choice(externalLinks)

def followExternalOnly(startingSite):
    externalLink = getRandomExternalLink(startingSite)
    print(f'Random external link is: {externalLink}')
    followExternalOnly(externalLink)


followExternalOnly('https://www.oreilly.com/')

HTTPError: HTTP Error 403: Forbidden

## Collect all External Links from a Site

In [71]:
# Collects a list of all external URLs found on the site
allExtLinks = []
allIntLinks = []

import time # Import the time module
import random # Import the random module for uniform delay

def getAllExternalLinks(url):
    # Add a User-Agent header to mimic a browser request
    req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    time.sleep(random.uniform(1, 3)) # Add a random delay between 1 and 3 seconds
    bs = BeautifulSoup(urlopen(req), 'html.parser')
    internalLinks = getInternalLinks(bs, url)
    externalLinks = getExternalLinks(bs, url)
    for link in externalLinks:
        if link not in allExtLinks:
            allExtLinks.append(link)
            print(link)

    for link in internalLinks:
        if link not in allIntLinks:
            allIntLinks.append(link)
            getAllExternalLinks(link)


allIntLinks.append('https://oreilly.com')
getAllExternalLinks('https://www.oreilly.com/')

HTTPError: HTTP Error 403: Forbidden